# Weather Alerts → Vector Embeddings Pipeline

End-to-end pipeline that fetches live weather alerts from the National Weather Service API,
stores them in Lakebase, and generates vector embeddings for semantic search.

**Pipeline Steps:**
1. **Fetch**: Pull active alerts from NWS API (`https://api.weather.gov/alerts/active/area/{state}`) for all 50 states
2. **Parse**: Transform GeoJSON responses into normalized schema
3. **Store**: Upsert alerts to `weather_information` table in Lakebase
4. **Chunk**: Split long narrative text using sliding window (size=800, overlap=100)
5. **Embed**: Generate 384-dim vectors using `sentence-transformers/all-MiniLM-L6-v2`
6. **Index**: Upsert embeddings to `weather_alert_embeddings` table with pgvector

**Features:**
- ✅ Always works with fresh, current alerts from NWS
- ✅ Rate-limited API calls (~1 req/sec)
- ✅ Idempotent upserts (safe to re-run)
- ✅ Same embedding model as ticker news for consistency

In [0]:
# Install required packages
%pip install -q 'databricks-sdk>=0.118.0' sentence-transformers pandas psycopg2

In [0]:
dbutils.library.restartPython()

## Config

Set table names and embedding parameters

In [0]:
# Table names (connecting to dataexpert_student database)
WEATHER_DOCS_TABLE = "weather_information"
EMBEDDINGS_TABLE = "weather_alert_embeddings"

# Chunking parameters
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

# Embedding model (same as ticker news for consistency)
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

print(f"Using model '{EMBEDDING_MODEL}' -> {EMBEDDING_DIM}-dim vectors")
print(f"Chunking: size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")

## Resolve Lakebase Connection

Parse connection details from the Databricks secret (same pattern as lakebase.py)

In [0]:
import os
from urllib.parse import urlparse

# Get Lakebase URL (dbutils.secrets returns raw value, not base64)
try:
    lakebase_url = dbutils.secrets.get(scope="database", key="lakebase-url")
    print("✅ Using dbutils.secrets")
except Exception:
    lakebase_url = os.environ.get("LAKEBASE_URL")
    if lakebase_url:
        print("✅ Using LAKEBASE_URL env var")
    else:
        raise ValueError("No Lakebase connection URL found")

# Parse connection details
parsed = urlparse(lakebase_url)
db_host = parsed.hostname
db_port = parsed.port or 5432
db_user = parsed.username
db_password = parsed.password

# IMPORTANT: Connect to dataexpert_student database for weather data
db_name = "dataexpert_student"

print(f"Connection: {db_user}@{db_host}:{db_port}/{db_name}")

In [0]:
import psycopg2

# List all available tables
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    # List all schemas
    cursor.execute("""
        SELECT schema_name 
        FROM information_schema.schemata 
        WHERE schema_name NOT IN ('pg_catalog', 'information_schema')
        ORDER BY schema_name
    """)
    schemas = [row[0] for row in cursor.fetchall()]
    print(f"📁 Available schemas: {schemas}\n")
    
    # List all tables
    cursor.execute("""
        SELECT table_schema, table_name 
        FROM information_schema.tables 
        WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
        ORDER BY table_schema, table_name
    """)
    tables = cursor.fetchall()
    
    print(f"📊 Found {len(tables)} tables:\n")
    for schema, table in tables:
        print(f"  {schema}.{table}")
        
finally:
    cursor.close()
    conn.close()

In [0]:
# Search for weather_information table
import psycopg2

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    # Search for weather_information in ANY schema
    cursor.execute("""
        SELECT table_schema, table_name 
        FROM information_schema.tables 
        WHERE LOWER(table_name) LIKE '%weather%' 
           OR LOWER(table_name) LIKE '%information%'
        ORDER BY table_schema, table_name
    """)
    tables = cursor.fetchall()
    
    if tables:
        print("🔍 Found tables matching 'weather' or 'information':")
        for schema, table in tables:
            full_name = f"{schema}.{table}"
            try:
                cursor.execute(f'SELECT COUNT(*) FROM "{schema}"."{table}"')
                count = cursor.fetchone()[0]
                print(f"  ✅ {full_name} ({count} rows)")
                
                # Show sample columns
                cursor.execute(f'''
                    SELECT column_name 
                    FROM information_schema.columns 
                    WHERE table_schema = %s AND table_name = %s
                    ORDER BY ordinal_position
                    LIMIT 10
                ''', (schema, table))
                columns = [row[0] for row in cursor.fetchall()]
                print(f"      Columns: {', '.join(columns[:5])}{'...' if len(columns) > 5 else ''}\n")
            except Exception as e:
                print(f"  ⚠️  {full_name} (could not read: {e})\n")
    else:
        print("❌ No tables found matching 'weather' or 'information'")
        print("\n💡 Try running: scripts/sync_weather_alerts.py to create and populate the table")
        
finally:
    cursor.close()
    conn.close()

In [0]:
import psycopg2

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    # Show current database
    cursor.execute("SELECT current_database()")
    current_db = cursor.fetchone()[0]
    print(f"🔌 Currently connected to database: {current_db}\n")
    
    # List ALL databases
    cursor.execute("""
        SELECT datname 
        FROM pg_database 
        WHERE datistemplate = false
        ORDER BY datname
    """)
    databases = [row[0] for row in cursor.fetchall()]
    print(f"📚 Available databases:")
    for db in databases:
        marker = "✅" if db == current_db else "  "
        print(f"{marker} {db}")
    
    # Check if dataexpert_student database exists
    if 'dataexpert_student' in databases:
        print(f"\n✅ Found 'dataexpert_student' database!")
        print(f"💡 Need to connect to it to see its tables")
    else:
        print(f"\n❌ 'dataexpert_student' database does not exist")
        print(f"💡 Run scripts/sync_weather_alerts.py to create it")
        
finally:
    cursor.close()
    conn.close()

In [0]:
import psycopg2

# Connect to the dataexpert_student database
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname='dataexpert_student',  # Connect to this database
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    print(f"📊 Tables in dataexpert_student database:\n")
    
    # List all schemas
    cursor.execute("""
        SELECT schema_name 
        FROM information_schema.schemata 
        WHERE schema_name NOT IN ('pg_catalog', 'information_schema', 'pg_toast')
        ORDER BY schema_name
    """)
    schemas = [row[0] for row in cursor.fetchall()]
    print(f"📁 Schemas: {schemas}\n")
    
    # List all tables with row counts
    cursor.execute("""
        SELECT table_schema, table_name 
        FROM information_schema.tables 
        WHERE table_schema NOT IN ('pg_catalog', 'information_schema', 'pg_toast')
        ORDER BY table_schema, table_name
    """)
    tables = cursor.fetchall()
    
    if tables:
        for schema, table in tables:
            full_name = f"{schema}.{table}"
            try:
                cursor.execute(f'SELECT COUNT(*) FROM "{schema}"."{table}"')
                count = cursor.fetchone()[0]
                print(f"  ✅ {full_name} ({count:,} rows)")
                
                # Show first few columns
                cursor.execute(f'''
                    SELECT column_name, data_type
                    FROM information_schema.columns 
                    WHERE table_schema = %s AND table_name = %s
                    ORDER BY ordinal_position
                    LIMIT 5
                ''', (schema, table))
                columns = cursor.fetchall()
                print(f"      Columns: {', '.join([f'{c[0]} ({c[1]})' for c in columns])}")
                if len(columns) == 5:
                    cursor.execute(f'''
                        SELECT COUNT(*) FROM information_schema.columns 
                        WHERE table_schema = %s AND table_name = %s
                    ''', (schema, table))
                    total_cols = cursor.fetchone()[0]
                    if total_cols > 5:
                        print(f"      ... and {total_cols - 5} more columns")
                print()
            except Exception as e:
                print(f"  ⚠️  {full_name} (error: {e})\n")
    else:
        print("  ❌ No tables found in dataexpert_student database")
        
finally:
    cursor.close()
    conn.close()

In [0]:
import psycopg2
import pandas as pd

# Connect to dataexpert_student database
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname='dataexpert_student',
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    # Get full schema
    cursor.execute("""
        SELECT column_name, data_type, character_maximum_length
        FROM information_schema.columns 
        WHERE table_schema = 'public' AND table_name = 'weather_information'
        ORDER BY ordinal_position
    """)
    columns = cursor.fetchall()
    
    print("📊 weather_information schema:\n")
    for col_name, data_type, max_length in columns:
        length_info = f"({max_length})" if max_length else ""
        print(f"  {col_name}: {data_type}{length_info}")
    
    # Get sample data
    print("\n👀 Sample rows:\n")
    df = pd.read_sql_query("SELECT * FROM weather_information LIMIT 3", conn)
    display(df)
    
    # Show row count
    cursor.execute("SELECT COUNT(*) FROM weather_information")
    count = cursor.fetchone()[0]
    print(f"\n📊 Total: {count} rows")
    
finally:
    cursor.close()
    conn.close()

## Fetch Fresh Weather Alerts from NWS API

Fetch current active alerts from the National Weather Service API for all US states and upsert to Lakebase.
This ensures we're always working with the latest data.

In [0]:
import requests
import time
from typing import List, Dict, Any

# US state codes to fetch alerts for
STATES = [
    'AL', 'AK', 'AZ', 'AR', 'CA', 'CO', 'CT', 'DE', 'FL', 'GA',
    'HI', 'ID', 'IL', 'IN', 'IA', 'KS', 'KY', 'LA', 'ME', 'MD',
    'MA', 'MI', 'MN', 'MS', 'MO', 'MT', 'NE', 'NV', 'NH', 'NJ',
    'NM', 'NY', 'NC', 'ND', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC',
    'SD', 'TN', 'TX', 'UT', 'VT', 'VA', 'WA', 'WV', 'WI', 'WY'
]

def fetch_state_alerts(state: str) -> List[Dict[str, Any]]:
    """Fetch active weather alerts for a US state from NWS API."""
    url = f"https://api.weather.gov/alerts/active/area/{state}"
    headers = {"User-Agent": "(Databricks Weather Embeddings, contact@example.com)"}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        return data.get('features', [])
    except Exception as e:
        print(f"  ⚠️  Error fetching {state}: {e}")
        return []

print(f"🌦️  Fetching active alerts from NWS API for {len(STATES)} states...\n")

all_alerts = []
for i, state in enumerate(STATES, 1):
    alerts = fetch_state_alerts(state)
    all_alerts.extend(alerts)
    
    if alerts:
        print(f"  [{i:2d}/{len(STATES)}] {state}: {len(alerts)} alerts")
    
    # Rate limiting: NWS asks for ~1 req/sec
    if i < len(STATES):
        time.sleep(1.1)

print(f"\n✅ Fetched {len(all_alerts)} total alerts from NWS API")

In [0]:
import json
from datetime import datetime
from typing import Optional

def parse_alert(feature: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Parse a GeoJSON alert feature into normalized schema."""
    try:
        props = feature.get('properties', {})
        
        # Extract key fields
        alert_id = props.get('id', '').replace('https://api.weather.gov/alerts/', '')
        if not alert_id:
            return None
        
        # Build location from areaDesc
        location = props.get('areaDesc', '')
        
        # Event type
        event = props.get('event', '')
        
        # Build narrative text from all text fields
        headline = props.get('headline', '')
        description = props.get('description', '')
        instruction = props.get('instruction', '')
        
        narrative_parts = []
        if event:
            narrative_parts.append(event)
        if headline:
            narrative_parts.append(headline + '.')
        if description:
            narrative_parts.append(description)
        if instruction:
            narrative_parts.append(f"INSTRUCTIONS: {instruction}")
        
        narrative_text = ' '.join(narrative_parts).strip()
        
        if not narrative_text:
            return None
        
        # Timestamps
        issued_at = props.get('sent')
        effective_at = props.get('effective')
        expires_at = props.get('expires')
        
        return {
            'id': alert_id,
            'location': location,
            'source_type': 'alert',
            'headline': headline,
            'event': event,
            'narrative_text': narrative_text,
            'issued_at': issued_at,
            'effective_at': effective_at,
            'expires_at': expires_at,
            'payload': json.dumps(feature),  # Store raw GeoJSON
            'synced_at': datetime.utcnow().isoformat() + 'Z'
        }
    except Exception as e:
        print(f"  ⚠️  Error parsing alert: {e}")
        return None

print(f"\n🔧 Parsing {len(all_alerts)} alerts...\n")

parsed_alerts = []
for alert in all_alerts:
    parsed = parse_alert(alert)
    if parsed:
        parsed_alerts.append(parsed)

print(f"✅ Parsed {len(parsed_alerts)} valid alerts")
print(f"   Skipped: {len(all_alerts) - len(parsed_alerts)} (missing required fields)")

if parsed_alerts:
    # Show sample
    sample = parsed_alerts[0]
    print(f"\n👀 Sample alert:")
    print(f"   ID: {sample['id'][:60]}...")
    print(f"   Location: {sample['location'][:80]}...")
    print(f"   Event: {sample['event']}")
    print(f"   Narrative length: {len(sample['narrative_text'])} chars")

In [0]:
import psycopg2
from psycopg2.extras import execute_values

if not parsed_alerts:
    print("⚠️  No alerts to upsert. Skipping.")
else:
    # Deduplicate alerts by ID (same alert can appear in multiple state API responses)
    unique_alerts = {alert['id']: alert for alert in parsed_alerts}
    parsed_alerts = list(unique_alerts.values())
    
    print(f"\n💾 Upserting {len(parsed_alerts)} unique alerts to weather_information...\n")
    
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        sslmode='require'
    )
    
    try:
        cursor = conn.cursor()
        
        # Ensure table exists
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS weather_information (
                id TEXT PRIMARY KEY,
                location TEXT,
                source_type TEXT,
                headline TEXT,
                event TEXT,
                narrative_text TEXT,
                issued_at TIMESTAMPTZ,
                effective_at TIMESTAMPTZ,
                expires_at TIMESTAMPTZ,
                payload JSONB,
                synced_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
            )
        """)
        print("✅ Table weather_information ready")
        
        # Prepare batch data
        values = [
            (
                alert['id'],
                alert['location'],
                alert['source_type'],
                alert['headline'],
                alert['event'],
                alert['narrative_text'],
                alert['issued_at'],
                alert['effective_at'],
                alert['expires_at'],
                alert['payload'],
                alert['synced_at']
            )
            for alert in parsed_alerts
        ]
        
        # Upsert with ON CONFLICT
        upsert_sql = """
            INSERT INTO weather_information (
                id, location, source_type, headline, event, narrative_text,
                issued_at, effective_at, expires_at, payload, synced_at
            ) VALUES %s
            ON CONFLICT (id) 
            DO UPDATE SET
                location = EXCLUDED.location,
                source_type = EXCLUDED.source_type,
                headline = EXCLUDED.headline,
                event = EXCLUDED.event,
                narrative_text = EXCLUDED.narrative_text,
                issued_at = EXCLUDED.issued_at,
                effective_at = EXCLUDED.effective_at,
                expires_at = EXCLUDED.expires_at,
                payload = EXCLUDED.payload,
                synced_at = EXCLUDED.synced_at
        """
        
        execute_values(cursor, upsert_sql, values)
        conn.commit()
        
        # Verify count
        cursor.execute("SELECT COUNT(*) FROM weather_information")
        total_count = cursor.fetchone()[0]
        
        print(f"✅ Upserted {len(parsed_alerts)} alerts")
        print(f"📊 Total alerts in weather_information: {total_count}")
        
    finally:
        cursor.close()
        conn.close()
    
    print(f"\n✨ Fresh weather data ready for embedding!")

## Load Unembedded Weather Documents

Read weather alerts from Lakebase that haven't been embedded yet

In [0]:
import pandas as pd
import psycopg2

print(f"Loading weather documents from {WEATHER_DOCS_TABLE}...")

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    # Query weather documents with narrative_text already built
    query = f"""
        SELECT 
            id,
            location,
            event,
            headline,
            narrative_text,
            effective_at,
            expires_at
        FROM {WEATHER_DOCS_TABLE}
        WHERE narrative_text IS NOT NULL 
          AND TRIM(narrative_text) != ''
    """
    
    weather_df = pd.read_sql_query(query, conn)
    print(f"✅ Loaded {len(weather_df)} weather documents")
    
    if len(weather_df) > 0:
        # Show text length distribution
        weather_df['text_length'] = weather_df['narrative_text'].str.len()
        print(f"\n📏 Text length stats:")
        print(f"  Min: {weather_df['text_length'].min()} chars")
        print(f"  Max: {weather_df['text_length'].max()} chars")
        print(f"  Mean: {weather_df['text_length'].mean():.0f} chars")
        print(f"  Median: {weather_df['text_length'].median():.0f} chars")
        print(f"\n📦 Documents > {CHUNK_SIZE} chars: {(weather_df['text_length'] > CHUNK_SIZE).sum()} (will be chunked)")
    
    display(weather_df.head(3))
    
finally:
    conn.close()

## Chunk Long Narrative Text

Split documents longer than CHUNK_SIZE using sliding window with overlap.
Short documents (most NWS alerts) are kept as single chunks.

In [0]:
print(f"Chunking narrative text (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})...")

out_doc_ids = []
out_locations = []
out_events = []
out_chunk_indexes = []
out_chunk_texts = []

for idx, row in weather_df.iterrows():
    doc_id = row['id']
    location = row['location']
    event = row['event']
    text = row['narrative_text']
    
    # Sliding window chunking
    if len(text) <= CHUNK_SIZE:
        # Short text - keep as single chunk
        out_doc_ids.append(doc_id)
        out_locations.append(location)
        out_events.append(event)
        out_chunk_indexes.append(0)
        out_chunk_texts.append(text.strip())
    else:
        # Long text - split with overlap
        chunk_idx = 0
        for start in range(0, len(text), CHUNK_SIZE - CHUNK_OVERLAP):
            chunk_text = text[start : start + CHUNK_SIZE].strip()
            if not chunk_text:
                continue
            
            out_doc_ids.append(doc_id)
            out_locations.append(location)
            out_events.append(event)
            out_chunk_indexes.append(chunk_idx)
            out_chunk_texts.append(chunk_text)
            
            chunk_idx += 1
            if start + CHUNK_SIZE >= len(text):
                break

chunks_df = pd.DataFrame({
    "document_id": out_doc_ids,
    "location": out_locations,
    "event": out_events,
    "chunk_index": out_chunk_indexes,
    "chunk_text": out_chunk_texts,
})

print(f"✅ Created {len(chunks_df)} chunks from {len(weather_df)} documents")
print(f"   Avg chunks per doc: {len(chunks_df) / len(weather_df):.2f}")

display(chunks_df.head(5))

## Compute Embeddings

Load sentence-transformers model and embed all chunks in batches

In [0]:
%pip install sentence-transformers

In [0]:
import os
from sentence_transformers import SentenceTransformer


# Set up HuggingFace cache
os.environ["HF_HOME"] = "/tmp/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/.cache/huggingface"

print(f"Loading embedding model {EMBEDDING_MODEL}...")
model = SentenceTransformer(EMBEDDING_MODEL, cache_folder="/tmp/.cache/huggingface")

# Compute embeddings in batches
print("Computing embeddings...")
batch_size = 32
all_embeddings = []

for i in range(0, len(chunks_df), batch_size):
    batch = chunks_df.iloc[i:i+batch_size]
    vectors = model.encode(batch["chunk_text"].tolist(), show_progress_bar=False)
    all_embeddings.extend(vectors.tolist())
    
    if (i + batch_size) % 128 == 0:
        print(f"  Processed {min(i + batch_size, len(chunks_df))}/{len(chunks_df)} chunks")

# Add embeddings to DataFrame
chunks_df['embedding'] = all_embeddings

print(f"✅ Computed {len(chunks_df)} embeddings ({EMBEDDING_DIM}-dim)")
print(f"   Sample embedding shape: {len(all_embeddings[0])} dimensions")

In [0]:
import psycopg2

print(f"Creating {EMBEDDINGS_TABLE} table if not exists...")

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    # Enable pgvector extension
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector")
    print("✅ pgvector extension enabled")
    
    # Create embeddings table
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {EMBEDDINGS_TABLE} (
            id TEXT PRIMARY KEY,
            document_id TEXT NOT NULL,
            chunk_index INTEGER NOT NULL,
            location TEXT,
            event TEXT,
            chunk_text TEXT NOT NULL,
            embedding vector(384),
            model_name TEXT NOT NULL,
            embedded_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
            UNIQUE(document_id, chunk_index)
        )
    """)
    print(f"✅ Created table {EMBEDDINGS_TABLE}")
    
    # Create indexes
    cursor.execute(f"""
        CREATE INDEX IF NOT EXISTS idx_weather_embeddings_location 
        ON {EMBEDDINGS_TABLE}(location)
    """)
    cursor.execute(f"""
        CREATE INDEX IF NOT EXISTS idx_weather_embeddings_event 
        ON {EMBEDDINGS_TABLE}(event)
    """)
    cursor.execute(f"""
        CREATE INDEX IF NOT EXISTS idx_weather_embeddings_doc 
        ON {EMBEDDINGS_TABLE}(document_id)
    """)
    print("✅ Created indexes")
    
    conn.commit()
    print(f"\n🎯 Table {EMBEDDINGS_TABLE} ready for embeddings!")
    
finally:
    cursor.close()
    conn.close()

## Upsert Embeddings to Lakebase

Batch insert embeddings with ON CONFLICT DO UPDATE for idempotency.

**Prerequisites:**
- Table `weather_alert_embeddings` must exist with pgvector extension
- Run this SQL in Lakebase first:

```sql
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS weather_alert_embeddings (
    id TEXT PRIMARY KEY,
    document_id TEXT NOT NULL,
    chunk_index INTEGER NOT NULL,
    location TEXT,
    event TEXT,
    chunk_text TEXT NOT NULL,
    embedding vector(384),
    model_name TEXT NOT NULL,
    embedded_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    UNIQUE(document_id, chunk_index)
);

CREATE INDEX IF NOT EXISTS idx_weather_embeddings_location ON weather_alert_embeddings(location);
CREATE INDEX IF NOT EXISTS idx_weather_embeddings_event ON weather_alert_embeddings(event);
CREATE INDEX IF NOT EXISTS idx_weather_embeddings_doc ON weather_alert_embeddings(document_id);
```

In [0]:
import psycopg2
from datetime import datetime
import json

print(f"Upserting {len(chunks_df)} embeddings into {EMBEDDINGS_TABLE}...")

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    # Prepare batch data with embeddings as JSON strings
    insert_data = []
    for idx, row in chunks_df.iterrows():
        embedding_id = f"{row['document_id']}_chunk_{row['chunk_index']}"
        embedding_json = json.dumps(row['embedding'])  # pgvector accepts JSON array string
        
        insert_data.append((
            embedding_id,
            row['document_id'],
            row['chunk_index'],
            row['location'],
            row['event'],
            row['chunk_text'],
            embedding_json,
            EMBEDDING_MODEL,
            datetime.now()
        ))
    
    # Batch upsert with ON CONFLICT UPDATE
    # Cast embedding to vector type in the INSERT using %s::vector
    upsert_sql = f"""
        INSERT INTO {EMBEDDINGS_TABLE} (
            id, document_id, chunk_index, location, event, chunk_text,
            embedding, model_name, embedded_at
        ) VALUES (%s, %s, %s, %s, %s, %s, %s::vector, %s, %s)
        ON CONFLICT (document_id, chunk_index) 
        DO UPDATE SET
            location = EXCLUDED.location,
            event = EXCLUDED.event,
            chunk_text = EXCLUDED.chunk_text,
            embedding = EXCLUDED.embedding,
            model_name = EXCLUDED.model_name,
            embedded_at = EXCLUDED.embedded_at
    """
    
    cursor.executemany(upsert_sql, insert_data)
    conn.commit()
    
    upserted_count = cursor.rowcount
    print(f"✅ Successfully upserted {upserted_count} embeddings to {EMBEDDINGS_TABLE}")
    print(f"   Embeddings stored as vector(384) type")
    
finally:
    cursor.close()
    conn.close()

print("\n✅ Weather alert embeddings ingestion complete!")

In [0]:
import psycopg2
import pandas as pd

print("\n🔍 Verifying embeddings in database...\n")

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)

try:
    cursor = conn.cursor()
    
    # Count embeddings
    cursor.execute(f"SELECT COUNT(*) FROM {EMBEDDINGS_TABLE}")
    count = cursor.fetchone()[0]
    print(f"📊 Total embeddings: {count}")
    
    # Count unique documents
    cursor.execute(f"SELECT COUNT(DISTINCT document_id) FROM {EMBEDDINGS_TABLE}")
    doc_count = cursor.fetchone()[0]
    print(f"📄 Unique documents: {doc_count}")
    print(f"🔢 Avg chunks per doc: {count / doc_count:.2f}")
    
    # Group by event type
    cursor.execute(f"""
        SELECT 
            event,
            COUNT(*) as embedding_count,
            COUNT(DISTINCT document_id) as document_count
        FROM {EMBEDDINGS_TABLE}
        GROUP BY event
        ORDER BY embedding_count DESC
        LIMIT 10
    """)
    event_stats = cursor.fetchall()
    
    print(f"\n📋 Top 10 event types:")
    for event, emb_count, doc_count in event_stats:
        print(f"  {event}: {emb_count} embeddings ({doc_count} docs)")
    
    # Verify vector dimensions
    cursor.execute(f"""
        SELECT 
            id,
            location,
            event,
            chunk_index,
            LEFT(chunk_text, 80) as preview,
            vector_dims(embedding) as dims
        FROM {EMBEDDINGS_TABLE}
        LIMIT 3
    """)
    
    print(f"\n👀 Sample embeddings:\n")
    df_sample = pd.read_sql_query(f"""
        SELECT 
            id,
            location,
            event,
            chunk_index,
            LEFT(chunk_text, 100) as chunk_preview,
            vector_dims(embedding) as embedding_dims
        FROM {EMBEDDINGS_TABLE}
        LIMIT 5
    """, conn)
    display(df_sample)
    
    print(f"\n✅ All embeddings verified! Ready for semantic search.")
    
finally:
    cursor.close()
    conn.close()